<a href="https://colab.research.google.com/github/Biegle-W/google_colab_llm/blob/main/notebooks/llama/llama_gpu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Colab llama.cpp hub

One authenticated OpenAI-compatible API for **llama.cpp** models on a **Google AI Pro / Colab Pro** GPU.
**This notebook installs only `llama`** and advertises only that engine's catalog ids.

- Supported models: GGUF: Qwen3.8-27B, Bolt IQ2, Sage, Ornith, Flash-Next / GLM / DeepSeek fallbacks
- Sibling notebooks: `notebooks/llama/`, `notebooks/vllm/`, `notebooks/sglang/`, `notebooks/diffusers/`, `notebooks/strata/`, `notebooks/maya/`, `notebooks/tensorfold/`, `notebooks/comfyui/`, `notebooks/sparsa/`, `notebooks/hyperqwen/`, `notebooks/ninfer/` — see `notebooks/llm_hub_colab.ipynb`
- Drive root: `MyDrive/llm_models/` (shared weights/builds; settings file is `config/settings_llama.json`)
- Engine matrix: `colab_llm_hub/ENGINE_MATRIX.md`

Runtime → Change runtime type → **L4, A100 or G4**, then run the cells in order.


## 1. Mount Drive, create the workspace, fetch (or restore) the hub code

Each run fetches the latest hub code from GitHub and refreshes its copy on Drive; only if GitHub cannot be reached does it fall back to that Drive copy. Settings and tool binaries are restored from Drive.


In [ ]:
import os, re, shutil, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")

ENGINE   = "llama"  # locked for this notebook
HUB_REPO  = "https://github.com/Biegle-W/google_colab_llm"
HUB_REF   = "main"  # always clone hub package from main
DRIVE_DIR = "/content/drive/MyDrive/llm_models"
HUB_DIR   = "/content/hub_repo"

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"{cmd}\n{r.stdout}{r.stderr}")
    return r.stdout.strip()

OFFLINE_HUB = False   # True = skip GitHub and use the hub code saved on Drive

os.makedirs(DRIVE_DIR, exist_ok=True)
sys.path.insert(0, HUB_DIR)
ref_key = (HUB_REF or "default").replace("/", "_")
cached = f"{DRIVE_DIR}/builds/hub_pkg/{ref_key}/colab_llm_hub"
have_cache = os.path.isdir(cached) and os.path.isfile(f"{DRIVE_DIR}/builds/hub_pkg/{ref_key}/COMPLETE")
fresh = False
if not OFFLINE_HUB:
    try:                          # always prefer the latest code: fixes land without reopening
        sh(f"rm -rf {HUB_DIR}")
        if HUB_REF and re.fullmatch(r"[0-9a-f]{40}", HUB_REF):
            sh(f"mkdir -p {HUB_DIR} && cd {HUB_DIR} && git init -q && git fetch -q --depth 1 {HUB_REPO} {HUB_REF} && git checkout -q FETCH_HEAD")
        else:
            sh(f"git clone -q --depth 1 {'--branch ' + HUB_REF if HUB_REF else ''} {HUB_REPO} {HUB_DIR}")
        fresh = True
        print("Fetched hub code from GitHub:", sh(f"cd {HUB_DIR} && git rev-parse --short HEAD"))
    except RuntimeError as e:
        print("Could not fetch from GitHub; using the copy on Drive.", str(e)[-300:])
if not fresh:
    if not have_cache:
        raise RuntimeError("No hub code: GitHub was unreachable and there is no copy on Drive yet.")
    shutil.rmtree(HUB_DIR, ignore_errors=True)
    os.makedirs(HUB_DIR, exist_ok=True)
    shutil.copytree(cached, f"{HUB_DIR}/colab_llm_hub")
    print("Restored hub package from Drive (it may be older than GitHub)")

from colab_llm_hub.workspace import open_workspace
from colab_llm_hub import install
WS = open_workspace(DRIVE_DIR, engine=ENGINE)
print("Drive workspace:", WS.describe())
# Release this runtime after this many minutes with no cell running, no API request, and no
# download / build / GPU work, so an idle VM stops using compute units (0 = never).
IDLE_TIMEOUT_MIN = 10  # @param {type:"integer"}
from colab_llm_hub import idle_guard
idle_guard.start(IDLE_TIMEOUT_MIN)
print(f"Idle timeout: {IDLE_TIMEOUT_MIN} min" if IDLE_TIMEOUT_MIN > 0 else "Idle timeout: off")
if fresh:                         # keep the Drive copy current for offline fallback
    install.cache_hub_package(HUB_DIR, str(WS.builds), ref=HUB_REF or "default")
import colab_llm_hub
print("hub", colab_llm_hub.__version__, "engine", ENGINE)


## 2. Detect GPU, CPU, and RAM — then list this engine's models


In [ ]:
from colab_llm_hub import catalog, gpu, recommend

G = gpu.detect()[0]
HOST = gpu.print_runtime(G)  # GPU + CPU + RAM + disk, every run
CPU, RAM = HOST["cpu"], HOST["ram"]
VRAM = G["vram_gb"]
ARCH = G["arch"]  # compute capability without the dot: 89 L4, 80 A100, 120 G4
RAM_GB = RAM["total_gb"]
from colab_llm_hub.store import model_disk_gb
DISK_GB = model_disk_gb(WS.local_models)   # free space + cached local models (they are evictable)
ALL = catalog.load()
ENTRIES = catalog.for_engines(ALL, [ENGINE])
PLAN = recommend.for_engine(ENGINE, VRAM, ALL, ram_gb=RAM_GB, disk_gb=DISK_GB, arch=ARCH)
print()
print(f"Models are read from Google Drive. Only SSD-streamed files go to {WS.local_models} "
      f"({DISK_GB} GB usable); fit checks use VRAM, RAM and that disk.")
print(recommend.explain(VRAM, ALL, engine=ENGINE, ram_gb=RAM_GB, disk_gb=DISK_GB, arch=ARCH))
print()
print(f"{'model id':36s} {'kind':5s} {'size':>7s}  {'ctx/edge':>14s}")
for e in ENTRIES:
    reason = catalog.why_not(e, VRAM, RAM_GB, DISK_GB, ARCH)
    fit = reason is None
    kind = catalog.kind_for(e)
    label = "edge" if kind in ("image", "video") else "ctx"
    ctx = f"{catalog.context_for(e, VRAM):,} {label}" if fit else "-"
    mark = " *" if e["id"] in PLAN["prefetch"] else ""
    print(f"{e['id']:36s} {kind:5s} {e['size_gb']:6.1f}G  {ctx:>14s}{mark}" + ("" if fit else f"   (does not fit: {reason})"))
print("\n* = recommended for this GPU + engine. Pick ONE model in the next cell.")


## 3. Config (saved on Drive under `config/settings_llama.json`)

**Pick one model** in the `MODEL` dropdown: it is the only model this session downloads and serves. `recommended` uses the first recommended model that fits this GPU. Other settings come from Drive; edit and re-run to persist.


In [ ]:
from colab_llm_hub import catalog, gpu, recommend

G = gpu.detect()[0]
HOST = gpu.print_runtime(G)  # re-detect GPU + CPU + RAM every config run
CPU, RAM = HOST["cpu"], HOST["ram"]
VRAM = G["vram_gb"]
ARCH = G["arch"]  # compute capability without the dot: 89 L4, 80 A100, 120 G4
RAM_GB = RAM["total_gb"]
from colab_llm_hub.store import model_disk_gb
DISK_GB = model_disk_gb(WS.local_models)   # free space + cached local models (they are evictable)
ALL = catalog.load()
ENTRIES = catalog.for_engines(ALL, [ENGINE])
PLAN = recommend.for_engine(ENGINE, VRAM, ALL, ram_gb=RAM_GB, disk_gb=DISK_GB, arch=ARCH)
saved = WS.load_settings()

# The ONE model this session downloads to Drive and serves. Nothing else is fetched.
MODEL = "recommended"  # @param ["recommended", "qwen3.8-27b-q4", "qwen3.8-27b-q8", "qwen3.8-27b-bf16", "qwen3.6-35b-a3b-iq4", "qwen3.6-35b-a3b-q8", "sage-a1.5-q4", "nemotron-3.5-super-120b-q3", "nemotron-3.5-super-120b-q4", "ornith-1.5-122b-q4", "ornith-1.5-122b-q5", "flash-next-iq1s", "flash-next-q3", "glm-5.3-flash-iq1s", "glm-5.3-flash-iq3", "deepseek-v4.1-flash-1017-iq1s", "nemotron-3.5-bolt-74b-iq2"]
SERVE_ONLY_SELECTED = True  # @param {type:"boolean"}
fitting = [e["id"] for e in ENTRIES if catalog.fits(e, VRAM, RAM_GB, DISK_GB, ARCH)]
if MODEL == "recommended":
    MODEL = (PLAN["prefetch"] or fitting or [None])[0]
    print("Recommended model for this GPU:", MODEL)
assert MODEL, f"No {ENGINE} model fits this GPU ({VRAM} GB); try a larger runtime."
assert MODEL in {e["id"] for e in ENTRIES}, f"{MODEL} is not a {ENGINE} model"
assert MODEL in fitting, (f"{MODEL} does not fit this runtime: "
                         f"{catalog.why_not(catalog.by_id(ENTRIES)[MODEL], VRAM, RAM_GB, DISK_GB, ARCH)}")
PREFETCH = [MODEL]
ENGINES = [ENGINE]  # never install other engines from this notebook
CTX_OVERRIDE = saved.get("ctx_override") or {}
LLAMA_CPP_REF = saved.get("llama_cpp_ref") or "4c3c5810f87efe177949f1b85ae509d38a2a311f"
REBUILD_LLAMA = bool(saved.get("rebuild_llama", False))
REFRESH_WHEELS = bool(saved.get("refresh_wheels", False))
# How your machine reaches this runtime (secrets live on private Drive under hub/, never in git):
#   tailscale        - private to your tailnet, stable https://colab-<engine>.<tailnet>.ts.net URL
#   cloudflare       - no account; public https://<words>.trycloudflare.com URL, new every session
#   cloudflare-named - your own Cloudflare tunnel: stable public hostname (set cf_hostname in settings)
TUNNEL = "tailscale"  # @param ["tailscale", "cloudflare", "cloudflare-named"]
CF_HOSTNAME = saved.get("cf_hostname") or ""  # cloudflare-named only, e.g. llm.example.com
REFRESH_CF_TOKEN = bool(saved.get("refresh_cloudflare_token", False))
PORT = int(saved.get("port") or 8000)
TS_HOSTNAME = saved.get("ts_hostname") or f"colab-{ENGINE}"
TS_FUNNEL = bool(saved.get("ts_funnel", False))  # False = tailnet-only Serve
REFRESH_TS_KEY = bool(saved.get("refresh_tailscale_key", False))

WS.save_settings({
    "engine": ENGINE, "model": MODEL, "serve_only_selected": SERVE_ONLY_SELECTED,
    "engines": ENGINES,
    "ctx_override": CTX_OVERRIDE, "llama_cpp_ref": LLAMA_CPP_REF,
    "rebuild_llama": REBUILD_LLAMA, "refresh_wheels": REFRESH_WHEELS,
    "tunnel": TUNNEL, "port": PORT, "hub_ref": HUB_REF,
    "ts_hostname": TS_HOSTNAME, "ts_funnel": TS_FUNNEL, "cf_hostname": CF_HOSTNAME,
    "refresh_cloudflare_token": False,
    "refresh_tailscale_key": False,  # set True once to re-prompt and overwrite Drive key
    "gpu_name": G["name"], "vram_gb": VRAM, "band": PLAN["band"],
    "cpu_model": CPU["model"], "cpu_cores": CPU["cores"], "cpu_threads": CPU["threads"],
    "ram_gb": RAM["total_gb"], "ram_available_gb": RAM["available_gb"],
})
print("Saved settings →", WS.settings_path)
print("MODEL    =", MODEL, "(the only model downloaded and served)" if SERVE_ONLY_SELECTED else "(prefetched; others load on request)")
print("ENGINE   =", ENGINE)
print("TUNNEL   =", TUNNEL)
print("Tailscale key file (private Drive, not git):", WS.hub / "tailscale_authkey")


## 4. Install llama.cpp (built or downloaded once, then restored from Drive)


In [ ]:
from colab_llm_hub import install

BUILDS = str(WS.builds)
LLAMA_BIN = "llama-server"
STRATA_BIN = "strata"
MAYA_ROOT = str(WS.builds / "project-maya")
TENSORFOLD_BIN = "tensorfold-native"
COMFYUI_ROOT = str(WS.builds / "ComfyUI")
SPARSA_ROOT = str(WS.builds / "Sparsa")
HYPERQWEN_ROOT = install.HYPERQWEN_ROOT
NINFER_BIN = install.NINFER_ROOT + "/apps/ninfer-serve"
install.ensure_hub_deps(BUILDS, refresh=REFRESH_WHEELS)
if ENGINE == "llama":
    LLAMA_BIN = install.ensure_llama(BUILDS, G["arch"], ref=LLAMA_CPP_REF, rebuild=REBUILD_LLAMA)
    print("llama-server:", LLAMA_BIN)
elif ENGINE == "vllm":
    print("vLLM", install.ensure_vllm(BUILDS, refresh=REFRESH_WHEELS))
elif ENGINE == "sglang":
    print("SGLang", install.ensure_sglang(BUILDS, refresh=REFRESH_WHEELS))
elif ENGINE == "diffusers":
    print("diffusers", install.ensure_diffusers(BUILDS, refresh=REFRESH_WHEELS))
elif ENGINE == "strata":
    STRATA_BIN = install.ensure_strata(BUILDS, refresh=REFRESH_WHEELS)
    print("strata:", STRATA_BIN)
elif ENGINE == "maya":
    MAYA_ROOT = install.ensure_maya(BUILDS, refresh=REFRESH_WHEELS)
    print("maya:", MAYA_ROOT)
elif ENGINE == "tensorfold":
    TENSORFOLD_BIN = install.ensure_tensorfold(BUILDS, refresh=REFRESH_WHEELS)
    print("tensorfold:", TENSORFOLD_BIN)
elif ENGINE == "comfyui":
    COMFYUI_ROOT = install.ensure_comfyui(BUILDS, refresh=REFRESH_WHEELS)
    print("comfyui:", COMFYUI_ROOT)
elif ENGINE == "sparsa":
    SPARSA_ROOT = install.ensure_sparsa(BUILDS, refresh=REFRESH_WHEELS)
    print("sparsa:", SPARSA_ROOT)
elif ENGINE == "hyperqwen":
    HYPERQWEN_ROOT = install.ensure_hyperqwen(BUILDS, refresh=REFRESH_WHEELS)
    print("hyperqwen:", HYPERQWEN_ROOT)
elif ENGINE == "ninfer":
    NINFER_BIN = install.ensure_ninfer(BUILDS, G["arch"], refresh=REFRESH_WHEELS)
    print("ninfer-serve:", NINFER_BIN)
print("Install log →", WS.logs / "install.log")


## 5. Download the models to Google Drive (skipped for anything already saved)

To spend less GPU time, run `notebooks/prepare/prepare_cpu.ipynb` on a CPU runtime first: it downloads the model and builds this engine on Drive, and this cell and cell 4 then just restore them.


In [ ]:
import threading, time
from colab_llm_hub.store import ModelStore

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

store = ModelStore(DRIVE_DIR, str(WS.local_models))
for mid in PREFETCH:
    e = catalog.by_id(ENTRIES)[mid]
    if store.state(e)["on_drive"]:
        print(f"{mid}: already on Drive")
        continue
    stop = threading.Event()
    def report(mid=mid, e=e):
        while not stop.wait(15):
            p = store.progress.get(mid, {})
            print(f"  {mid}: {p.get('phase', '')} {p.get('done_bytes', 0) / 1e9:.1f} / {e['size_gb']:.1f} GB")
    threading.Thread(target=report, daemon=True).start()
    t0 = time.time()
    try:
        store.prefetch(e)
    finally:
        stop.set()
    print(f"{mid}: saved to Drive in {(time.time() - t0) / 60:.1f} min")


## 6. Start the hub and open the tunnel

Re-running this cell restarts the hub. Secrets stay on **your private Google Drive** (`MyDrive/llm_models/hub/`), never in this public GitHub repo or in settings JSON:

- `hub/api_key` — hub Bearer token (auto-created)
- `hub/tailscale_authkey` — Tailscale auth key (prompted once, then reused)
- `hub/cloudflare_tunnel_token` — only for `cloudflare-named` (prompted once, then reused)

**Tailscale** (default): create a **reusable + ephemeral** key at [Tailscale Keys](https://login.tailscale.com/admin/settings/keys) and join the machine your harness runs on to the same tailnet. Set `refresh_tailscale_key` in settings to replace the key.

**Cloudflare** (`TUNNEL = "cloudflare"`): nothing to set up, like Unsloth Studio's notebook. The URL is public and changes every session, so update your harness each time; the hub's API key is what keeps others off your GPU. `cloudflare-named` uses a tunnel you created in the Cloudflare dashboard (public hostname → `http://127.0.0.1:8000`) for a stable URL.

Your harness runs on **your own machine**, not here: point it at the Base URL this cell prints.


In [ ]:
import json, requests, time
from colab_llm_hub import keyfile, tunnel, provider

subprocess.run("pkill -TERM -f '[c]olab_llm_hub serve'; pkill -TERM -f '[t]ailscaled.*colab-llm-tailscale'", shell=True)
time.sleep(3)

API_KEY = keyfile.load_or_create(str(WS.hub / "api_key"))
HUB_LOG = str(WS.logs / f"hub_{ENGINE}.log")
args = [sys.executable, "-m", "colab_llm_hub", "serve", "--drive-dir", DRIVE_DIR,
        "--local-dir", str(WS.local_models), "--log-dir", str(WS.engine_logs),
        "--port", str(PORT), "--llama-bin", LLAMA_BIN,
        "--strata-bin", STRATA_BIN, "--maya-root", MAYA_ROOT,
        "--tensorfold-bin", TENSORFOLD_BIN, "--comfyui-root", COMFYUI_ROOT,
        "--sparsa-root", SPARSA_ROOT,
        "--hyperqwen-root", HYPERQWEN_ROOT, "--ninfer-bin", NINFER_BIN,
        "--engines", ENGINE]
if SERVE_ONLY_SELECTED:
    args += ["--models", MODEL]   # the harness cannot trigger downloads of other models
for k, v in CTX_OVERRIDE.items():
    args += ["--ctx", f"{k}={v}"]
hub_proc = subprocess.Popen(args, cwd=HUB_DIR, env=dict(os.environ, PYTHONPATH=HUB_DIR, COLAB_LLM_DRIVE=DRIVE_DIR),
                            stdout=open(HUB_LOG, "w"), stderr=subprocess.STDOUT)

AUTH = {"Authorization": f"Bearer {API_KEY}"}
for _ in range(60):
    try:
        if requests.get(f"http://127.0.0.1:{PORT}/health", headers=AUTH, timeout=2).status_code == 200:
            break
    except requests.RequestException:
        pass
    if hub_proc.poll() is not None:
        raise RuntimeError("The hub exited:\n" + open(HUB_LOG).read()[-3000:])
    time.sleep(1)
else:
    raise RuntimeError("The hub did not start:\n" + open(HUB_LOG).read()[-3000:])

if TUNNEL == "tailscale":
    # Load the auth key from private Drive, or prompt once and save it there (never committed to git).
    TS_AUTHKEY = tunnel.load_or_ask_tailscale_authkey(str(WS.hub / "tailscale_authkey"),
                                                      refresh=REFRESH_TS_KEY)
    tunnel_proc, BASE = tunnel.start_tailscale(
        PORT, TS_AUTHKEY, hostname=TS_HOSTNAME, drive_dir=DRIVE_DIR, funnel=TS_FUNNEL)
    del TS_AUTHKEY  # drop from notebook namespace; Drive file remains private
elif TUNNEL == "cloudflare":
    tunnel_proc, BASE = tunnel.start_cloudflare_quick(PORT, drive_dir=DRIVE_DIR, api_key=API_KEY)
elif TUNNEL == "cloudflare-named":
    CF_TOKEN = tunnel.load_or_ask_secret(
        str(WS.hub / "cloudflare_tunnel_token"), "Cloudflare tunnel token (Zero Trust → Networks → "
        "Tunnels → your tunnel → install command): ", refresh=REFRESH_CF_TOKEN, env_name="CLOUDFLARE_TUNNEL_TOKEN")
    tunnel_proc, BASE = tunnel.start_cloudflare_named(PORT, CF_TOKEN, CF_HOSTNAME, drive_dir=DRIVE_DIR,
                                                      api_key=API_KEY)
    del CF_TOKEN
else:
    raise ValueError(f"Unknown TUNNEL {TUNNEL!r}")

MODELS = requests.get(f"http://127.0.0.1:{PORT}/v1/models", headers=AUTH).json()["data"]
WS.save_endpoint(BASE + "/v1", API_KEY, G["name"], [m["id"] for m in MODELS])

print("Engine   :", ENGINE)
print("Tunnel   :", TUNNEL)
print("Base URL :", BASE + "/v1")
print("API key  :", API_KEY)
print(f"Models   : {', '.join(m['id'] for m in MODELS)}")
print("Endpoint →", WS.hub / "endpoint.json")
if TUNNEL == "tailscale" and not TS_FUNNEL:
    print("Reach this URL from any device on your Tailscale tailnet (e.g. the machine your harness runs on).")
elif TUNNEL == "cloudflare":
    print("Public URL, new every session: update your harness's base URL each time. The API key protects it.")
print()
print("Any OpenAI-compatible client:")
print(provider.openai_env(BASE, API_KEY))
print("dsh (runs on YOUR machine, not in this notebook): in a clone of this repo there, run")
print(f"  python -m colab_llm_hub.configure_dsh --url {BASE} --key {API_KEY} --settings <path to dsh settings.yaml>")
print(f"  export COLAB_LLM_API_KEY={API_KEY}\n")
print(provider.dsh_provider_yaml(BASE, MODELS))


## 7. Try chat

The first request to a model loads it (download or Drive→local copy, then GPU load). Streaming chat waits with keep-alives; non-streaming may return `503` + `Retry-After` while loading — re-run the cell.


In [ ]:
MODEL = next((m for m in PREFETCH if m in {x["id"] for x in MODELS}), MODELS[0]["id"])
messages = [{"role": "user", "content": "In two sentences, what is a model hub?"}]
t0 = time.time()
r = requests.post(f"http://127.0.0.1:{PORT}/v1/chat/completions", headers=AUTH, timeout=900,
                  json={"model": MODEL, "max_tokens": 200, "messages": messages})
if r.status_code == 503 and r.json().get("error", {}).get("code") == "model_loading":
    print("Chat model still loading; re-run this cell in a minute.")
else:
    j = r.json()
    text = j["choices"][0]["message"]["content"] if r.ok else j
    print("=== chat:", MODEL, "===")
    print(text)
    print(f"[{time.time() - t0:.1f} s]")
    if r.ok:
        path = WS.save_chat(MODEL, messages, text, j.get("usage"))
        print("Saved →", path)


## 8. Status and logs (re-run any time)


In [ ]:
st = requests.get(f"http://127.0.0.1:{PORT}/v1/hub/status", headers=AUTH).json()
WS.save_json(WS.status, f"hub-status-{ENGINE}", st)
print(json.dumps({k: st[k] for k in ("gpu", "vram_gb", "active", "loading", "phase", "inflight", "errors", "local_free_gb")}, indent=1))
for mid, p in st["downloads"].items():
    print(f"  {mid}: {p.get('phase')} {p.get('done_bytes', 0) / 1e9:.1f} / {p.get('total_bytes', 0) / 1e9:.1f} GB")
print("\n--- hub log (Drive) ---"); print(open(HUB_LOG).read()[-1500:])
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv
print("Outputs →", WS.outputs)


## 9. Stop


In [ ]:
hub_proc.terminate()        # stops the hub and the engine it started
try:
    tunnel_proc.terminate()
except Exception:
    pass
if TUNNEL == "tailscale":
    tunnel.stop_tailscale(drive_dir=DRIVE_DIR)
else:
    tunnel.stop_cloudflare()


## Notes

### This notebook = `llama` only

| Catalog id | Notes |
|---|---|
| `qwen3.8-27b-q4` / `q8` / `bf16` | Qwen3.8-27B GGUF |
| `nemotron-3.5-bolt-74b-iq2` | Native IQ2 Bolt (workstation selected) |
| `sage-a1.5-q4`, `ornith-1.5-122b-q4` | Sage / Ornith on A100–G4 |
| `flash-next-iq1s` / `flash-next-q3` | Flash-Next fallback (prefer **Strata**) |
| `glm-5.3-flash-iq1s` / `iq3` | GLM fallback (prefer **Maya**) |
| `deepseek-v4.1-flash-1017-iq1s` | DeepSeek fallback (prefer **Maya** / **Strata**) |

### Connecting your harness

**Tailscale** (`TUNNEL = "tailscale"`, default, private):

1. Create a **reusable + ephemeral** auth key: https://login.tailscale.com/admin/settings/keys
2. First run: cell 6 prompts for it and saves to **private Drive** `MyDrive/llm_models/hub/tailscale_authkey` (not GitHub).
3. Later Colab sessions reuse that Drive file automatically.
4. Optional Colab secret `TAILSCALE_AUTHKEY` if you press Enter on an empty prompt.
5. Join the machine your harness runs on to the same tailnet and point the harness at the printed `https://….ts.net/v1` URL with the API key. The harness (dsh or any OpenAI client) runs there, never in this notebook.

**Cloudflare quick tunnel** (`TUNNEL = "cloudflare"`): no account or key. Cell 6 prints a public `https://….trycloudflare.com/v1` URL once the hub answers through it; it changes every session, and anyone with it reaches the hub, which still requires the API key. Quick tunnels are a free best-effort Cloudflare service with no uptime guarantee.

**Cloudflare named tunnel** (`TUNNEL = "cloudflare-named"`): create a tunnel in the Cloudflare dashboard (Zero Trust → Networks → Tunnels), add a public hostname pointing to `http://127.0.0.1:8000`, set `cf_hostname` in this engine's settings file, and paste the tunnel token when cell 6 asks (saved to `hub/cloudflare_tunnel_token` on private Drive).

### Drive layout

| Path under `MyDrive/llm_models` | Contents |
|---|---|
| `models/<id>/` | Weights shared across engine notebooks |
| `builds/` | llama.cpp tarballs, wheels, hub_pkg, Tailscale, cloudflared |
| `config/settings_llama.json` | Selected model / ctx / Tailscale for **this** engine (no secrets) |
| `hub/api_key` | Hub Bearer token (private Drive only) |
| `hub/tailscale_authkey` | Tailscale auth key (private Drive only; never in git) |
| `hub/cloudflare_tunnel_token` | Cloudflare named-tunnel token (private Drive only; never in git) |
| `hub/endpoint.json` | Last base URL + key hint |
| `hf_cache/`, `logs/`, `outputs/` | Cache, logs, generation results |

- See `colab_llm_hub/ENGINE_MATRIX.md` for workstation (Maya / Strata / HyperQwen) notes.
- Set `REFRESH_WHEELS` / `REBUILD_LLAMA` / `refresh_tailscale_key` in settings to force updates.
- **Security:** Tailscale Serve is reachable only from your tailnet (`ts_funnel` makes it public); Cloudflare URLs are public. Either way the hub API key protects GPU use. Never commit anything under `hub/`.
